# Algebraic Reduction and Numerical Optimization Methods for solving Quantum Control Sum of Squares Problems

This research is a work in progress. You can find the preprint IEEE paper at https://alexanderleong.com/public/qcsos_paper.pdf. This notebook is meant to complement the details described in the paper. As the solver continues to improve you should expect the quality of the results to vary. If you have problems running the package, notebook or if you find errors, contact the author directly by email at toshibaalexander@gmail.com.

**IMPORTANT NOTES** \
You must run with Julia 1.11 (2025-07). Later versions yet to be supported. \
The current Julia package ConicSolve.jl is out of date. You must use the latest version in the GitHub repo.

## Background

This article explores algebraic methods for quantum control polynomial optimization problems. We show how degeneracy, numerical stability and scalability are central issues in quantum control and how ConicSolve.jl has been extended to use face reduction and symmetry reduction (via the Wedderburn decomposition).

Many quantum control problems are posed as polynomial optimization problems. Sum of Squares programming is effective at utilizing symmetric self-dual solvers due to such solvers exhibiting polynomial-time convergence and achieving solutions to high precision. Despite this solving Semidefinite programs and moreover Sum of Squares programs remain challenging. First is the computational blowup in memory and runtime cost as the problem size increases, second is the ill conditioning of solving polynomial optimization problems using the monomial basis, third is the numerical struggle that solvers must deal with.

Sum of squares programming is widely used in control (Lyapunov stability), robotics (trajectory optimization) and the validation and verification of safety critical systems. Global convergence and solution quality are becoming increasingly relevant in quantum control and quantum information sciences. Scaling quantum systems without compromising solution quality remains an ongoing challenge. This motivates the question; how can we adapt existing well established algebraic and computational tools in the quantum realm.

In this article we’ll focus on the fixed-time control problem of finding the control that achieves as close as possible a given target unitary at the end of a given evolution time. The QCPOP method popularized by Bondar et al., 2025 uses TSSOS.jl for solving the sum of squares programming problem. We take a different approach by proposing a computational pipeline based on the ideas discussed in Permenter, 2017 to apply face reduction on top of symmetry reduction as a preprocessing step before solving the SOS problem. This is attractive for several theoretical/practical reasons, it exploits problem structure without relaxing the original problem, these mathematical methods can act as additional layers to extend existing solvers, packages such as SymbolicWedderburn.jl (which we use) already exists for exploiting algebraic structure.

An ongoing challenge with adopting these tools is a robust solver architecture. We’ll show the changes in ConicSolve.jl made to further reduce degeneracy and numerical ill-conditioning, particularly the use of matrix decompositions, equilibration and regularization methods. We hope this article stimulates discussion and interest in further advancing Julia’s ecosystem to realize and explore new algebraic and computational methods. It is without many Julia packages like OperatorScaling.jl and SymbolicWedderburn.jl that this article and exploration of these ideas would remain just that, a theoretical ideal.

To get started with QCSOS.jl we need to import a few packages

In [ ]:
run(`git clone git@github.com:alexander-leong/ConicSolve.jl.git@main`)
run(`git clone git@github.com:alexander-leong/QCSOS.jl.git@test`)

if Int64(VERSION.major) != 1 || Int64(VERSION.minor) != 11
    @warn "Unsupported Julia major/minor version!"
else
    using Pkg
    Pkg.add("/content/ConicSolve.jl/src")
    Pkg.add("/content/ConicSolve.jl/ext/experimental/ConicSolveFR.jl/src")
    Pkg.add("/content/QCSOS.jl/src")
    Pkg.activate("/content/QCSOS.jl")
end

In [ ]:
using DynamicPolynomials
using LinearAlgebra
using ConicSolve
using QCSOS
using Random

## The fixed time control problem

The fixed time control problem involves finding the control coefficients in ``E(t)`` that results in a unitary ``U(T)`` which is as close as possible to a given target unitary, ``U*`` at time ``T``.

Specifically we solve the optimization problem
$$ \begin{aligned}\text{minimize}\qquad & Tr(M^{\dagger}M) \\
\text{subject to}\qquad & \partial_tU(t) = A(t)U(t) \\
\qquad & U(0) = \mathbb{1} \\
\qquad & A(t) = -i(H_0 + E(t)V) \end{aligned} $$

where the objective function in Eq. (1) is approximated as
$$ \|e^{\Lambda_n/2} - e^{-\Lambda_n/2}U^*\|^2 $$

and the control function takes the form
$$ E(t) = \Sigma_{k=0}^{m-1}x_kt^k $$

where $x_k$ are the coefficients being solved for.

Below we define the problem variables.

In [5]:
i = 1
T = 0.5
# Drift Hamiltonian
H0 = [
    0 0 0;
    0 3.21505101e+10 0;
    0 0 6.23173079e+10
];

H0 ./= norm(H0, Inf);

# Control Hamiltonian
V = [
    0 1 0;
    1 0 1.41421356;
    0 1.41421356 0
];

V ./= norm(V, Inf);

n_samples = 1000
Random.seed!(6292022)
problem = QuantumControlSOSProblem(H0, V, T, degree)
exact_x = -1 .+ 2 * rand(length(problem.x) * n_samples)
exact_x = reshape(exact_x, (length(problem.x), n_samples))
U_target = get_unitary(problem, exact_x[:, i])

3×3 Matrix{ComplexF64}:
   0.975721+0.00204083im  0.0281255+0.214463im  -0.0331948+0.00880109im
  0.0266327+0.214653im     0.896927-0.234773im    0.114424+0.283769im
 -0.0333103+0.00835158im   0.112561+0.284513im    0.833099-0.459509im

We can approximate the solution to the initial value problem of the ODE
$$ \partial_tU(t) = A(t)U(t), U(0) = 1 $$
using the Magnus expansion, for the first three terms this is \
$ \Omega_1(t) = \int_0^t A(t_1) dt_1 $, \
$ \Omega_2(t) = 1/2! \int_0^t dt_1 \int_0^{t_1} dt_2[A(t_1), A(t_2)] $, \
$ \Omega_3(t) = 1/3! \int_0^t dt_1 \int_0^{t_1} dt_2 \int_0^{t_2} dt_3([A(t_1),[A(t_2),A(t_3)]] + [A(t_3),[A(t_2),A(t_1)]]) $

$$U(T) = \lim\limits_{n\rightarrow\infin} \text{exp}[\displaystyle\sum_{k=1}^n\Omega_k]$$

The expression of this problem is handled internally by QCSOS.jl, so we proceed to define the Fixed Time Problem and hit solve.

In [6]:
ϵ = 1e-4
η_eps = 1e-2
η_lambda = 1e-4
p = 3
problem = QuantumUnitaryFixedTimeProblem(U_target, problem, degree, p)
@info "Optimization problem constructed successfully."

qcsos_solver = QCSOS_Solver(p, problem, ϵ, η_eps, η_lambda)
_, reduced_solvers = solve!(qcsos_solver)

┌ Info: ### Executing Symmetry Reduction ###
└ @ ConicSolve /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/src/models/sos_sg.jl:138
┌ Info: Polynomial function f has degree: 12
└ @ ConicSolve /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/src/models/sos_sg.jl:139
┌ Info: Symmetric Group 4
└ @ ConicSolve /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/src/models/sos_sg.jl:140
┌ Info: End of decompose, returning...
└ @ ConicSolve /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/src/models/sos_sg.jl:183
┌ Info: Optimization problem constructed successfully.
└ @ Main /home/alexander/Documents/alexander_leong/projects/QCSOS.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W5sdnNjb2RlLXJlbW90ZQ==.jl:6
┌ Info: Performing RRQR regularization
└ @ ConicSolve /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/src/frontend/interface.jl:129
┌ Info: ### Executing face reduction ###
└ @ ConicSolveFR /home

[ Info: Optimize called
[ Info:                Solver parameters 
[ Info:                Objective limit:  -Inf
[ Info:                 Limit solution:  0.0
[ Info:                 Max iterations:  4
[ Info:                Num constraints:  530
[ Info:                    Num threads:  1
[ Info:               Preferred device:  CPU
[ Info: Iterative Refinement Max Iterations:  1
[ Info: Iterative Refinement Trigger Mode:  NO_ITERATIVE_REFINEMENT
[ Info:  Presolve Regularization Method: none
[ Info: Presolve Scaling (Equilibration) Method: ruiz
[ Info:       System Solver KKT method:  QR and Cholesky
[ Info:           Time limit (seconds):  1000000
[ Info:         Tolerance gap absolute:  0.0001
[ Info:         Tolerance gap relative:  0.0001
[ Info:           Tolerance optimality:  0.01
[ Info: Performing Ruiz Equilibration
[ Info: Solving for primal dual starting points
[ Info: Executing main optimization loop
[ Info: Iter. |      Dual Res. |    Primal Res. |     Cent. Res. |      Dual

┌ Info: Iter. |      Dual Res. |    Primal Res. |     Cent. Res. |      Dual. Gap |    Primal Obj. |      Dual Obj. |      Step size |            b'y |
└ @ ConicSolve /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/src/status.jl:142
┌ Info:     4 |        3.28712 |    1.19967e-15 |        201.702 |       0.529364 |            0.0 |     -0.0855058 |       0.105579 |      0.0929414 |
└ @ ConicSolve /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/src/status.jl:152
┌ Info: WARNING: gap s'z 0.09294144136363204 above threshold 0.01
└ @ ConicSolveFR /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/ext/experimental/ConicSolveFR.jl/src/ConicSolveFR.jl:154
┌ Info: Constraint already feasible, nothing to reduce
└ @ ConicSolveFR /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/ext/experimental/ConicSolveFR.jl/src/ConicSolveFR.jl:97
┌ Info: Nothing reduced
└ @ ConicSolveFR /home/alexander/Documents/alexander_leong/projects/ConicSolve

[ Info: Slow progress...
[ Info:     3 |        3.26389 |    9.71264e-16 |        46161.2 |        0.53782 |            0.0 |    -0.00401218 |    0.000604886 |
[ Info:     4 |         3.2832 |    1.03959e-15 |        200.696 |       0.529535 |            0.0 |     -0.0863251 |       0.106779 |
┌ Info: Solver finished
│ Exit status: ITERATION_LIMIT
│ Primal objective value: 0.0
│ Number of iterations: 4
└ Time elapsed: 0.15722953099999998
[ Info: Optimize called
[ Info:                Solver parameters 
[ Info:                Objective limit:  -Inf
[ Info:                 Limit solution:  0.0
[ Info:                 Max iterations:  4
[ Info:                Num constraints:  211
[ Info:                    Num threads:  1
[ Info:               Preferred device:  CPU
[ Info: Iterative Refinement Max Iterations:  1
[ Info: Iterative Refinement Trigger Mode:  NO_ITERATIVE_REFINEMENT
[ Info:  Presolve Regularization Method: none
[ Info: Presolve Scaling (Equilibration) Method: ruiz
[ Info:  

┌ Info: Iter. |      Dual Res. |    Primal Res. |     Cent. Res. |      Dual. Gap |    Primal Obj. |      Dual Obj. |      Step size |            b'y |
└ @ ConicSolve /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/src/status.jl:142
┌ Info:     4 |         3.2832 |    1.03959e-15 |        200.696 |       0.529535 |            0.0 |     -0.0863251 |       0.106779 |      0.0904817 |
└ @ ConicSolve /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/src/status.jl:152
┌ Info: Solved reduced problem
└ @ ConicSolveFR /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/ext/experimental/ConicSolveFR.jl/src/ConicSolveFR.jl:282
┌ Info: Reproject iterate
└ @ ConicSolveFR /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/ext/experimental/ConicSolveFR.jl/src/ConicSolveFR.jl:288
┌ Info: Reproject iterate
└ @ ConicSolveFR /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/ext/experimental/ConicSolveFR.jl/src/ConicSolveFR.jl:288


...

[ Info: Iter. |      Dual Res. |    Primal Res. |     Cent. Res. |      Dual. Gap |    Primal Obj. |      Dual Obj. |      Step size |
[ Info:     1 |        6.17401 |    1.95532e-20 |     4.50708e-6 |        1.01522 |            0.0 |    0.000225076 |       0.163258 |
[ Info:     2 |        2.55809 |    1.35028e-15 |        3.10099 |        0.42076 |            0.0 |    -0.00843314 |        0.62856 |
[ Info: Slow progress...
[ Info:     3 |        2.97071 |    1.28657e-15 |        24988.7 |       0.420256 |            0.0 |    0.000212938 |    0.000781487 |
[ Info: Slow progress...
[ Info:     4 |         3.2072 |    1.27816e-15 |        1400.37 |         0.4197 |            0.0 |   -0.000291583 |     0.00358229 |
┌ Info: Solver finished
│ Exit status: ITERATION_LIMIT
│ Primal objective value: 0.0
│ Number of iterations: 4
└ Time elapsed: 0.803201901
[ Info: Optimize called
[ Info:                Solver parameters 
[ Info:                Objective limit:  -Inf
[ Info:                 

┌ Info: Iter. |      Dual Res. |    Primal Res. |     Cent. Res. |      Dual. Gap |    Primal Obj. |      Dual Obj. |      Step size |            b'y |
└ @ ConicSolve /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/src/status.jl:142
┌ Info:     4 |         3.2072 |    1.27816e-15 |        1400.37 |         0.4197 |            0.0 |   -0.000291583 |     0.00358229 |     0.00807909 |
└ @ ConicSolve /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/src/status.jl:152
┌ Info: Solved reduced problem
└ @ ConicSolveFR /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/ext/experimental/ConicSolveFR.jl/src/ConicSolveFR.jl:282
┌ Info: Reproject iterate
└ @ ConicSolveFR /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/ext/experimental/ConicSolveFR.jl/src/ConicSolveFR.jl:288
┌ Info: Reproject iterate
└ @ ConicSolveFR /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/ext/experimental/ConicSolveFR.jl/src/ConicSolveFR.jl:288


┌ Info: Solver finished
│ Exit status: ITERATION_LIMIT
│ Primal objective value: 0.0
│ Number of iterations: 4
└ Time elapsed: 0.013083831


┌ Info: Iter. |      Dual Res. |    Primal Res. |     Cent. Res. |      Dual. Gap |    Primal Obj. |      Dual Obj. |      Step size |            b'y |
└ @ ConicSolve /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/src/status.jl:142
┌ Info:     4 |        1.44543 |    6.23237e-16 |        25.8466 |       0.276814 |            0.0 |     -1.7389e-9 |       0.182095 |     4.5288e-11 |
└ @ ConicSolve /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/src/status.jl:152
┌ Info: Solved reduced problem
└ @ ConicSolveFR /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/ext/experimental/ConicSolveFR.jl/src/ConicSolveFR.jl:282
┌ Info: Reproject iterate
└ @ ConicSolveFR /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/ext/experimental/ConicSolveFR.jl/src/ConicSolveFR.jl:288
┌ Info: Reproject iterate
└ @ ConicSolveFR /home/alexander/Documents/alexander_leong/projects/ConicSolve.jl/ext/experimental/ConicSolveFR.jl/src/ConicSolveFR.jl:288


(Any[[0.10496759296404279, 3.117110645685742e-6, -0.06959916321635147, -0.014786293177617146, 0.0015948522455750463, 0.0001355375376592535, 0.00013402858841027592, -1.1733398964069622e-5, 5.271499688104014e-7, 2.139263051651824e-7  …  -4.4270694274529983e-20, 4.046861285549736e-21, -5.275567302322884e-20, 6.481596837721393e-20, 7.214884566027975e-20, -3.2467598342586905e-20, 1.645096051052204e-20, -2.390868415412615e-19, -2.649503925621388e-19, -9.856378093971404e-20], [-1.4507619906474471e-6, -2.0820115142113244e-9, -8.955653320554076e-16, 2.6229669603664837e-16, 1.354501883348349e-7, -9.902820249412034e-10, 2.0603185807682536e-9, 1.2438213958587089e-11, -1.6216331943034303e-11, 6.3421524489182685e-12  …  3.061372962420494e-27, -1.036890174263309e-26, 2.8393390794354834e-26, 6.018618336989848e-27, 6.17708035504897e-27, -4.320758124420076e-26, 4.5242484935795354e-26, 7.538750213626189e-26, 5.604243851890941e-26, 2.646277235215036e-26], [0.014878327308389953, -0.0019154046383895835, -9.

# Evaluating the solution
To evaluate the solution we first obtain the control coefficients. From this we can then determine the infidelity achieved by comparing the target unitary with the estimated unitary returned by the solver.

In [7]:
# get solution
solution = get_solution(problem.program, qcsos_solver.x_vec)

# evaluate solution
coefficients = get_control(problem, solution)
@info("Extracted control coefficients")
H_result = get_unitary_from_control(problem, coefficients)
infidelity = get_infidelity(U_target, H_result)
@info("Infidelity: $(infidelity)")

┌ Info: Numerical rank of solution (ideally 1) with tolerance 0.0001 is 5
└ @ QCSOS /home/alexander/Documents/alexander_leong/projects/QCSOS.jl/src/solver.jl:74
┌ Info: Extracted control coefficients
└ @ Main /home/alexander/Documents/alexander_leong/projects/QCSOS.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sdnNjb2RlLXJlbW90ZQ==.jl:6
┌ Info: Infidelity: 0.10189508452900675
└ @ Main /home/alexander/Documents/alexander_leong/projects/QCSOS.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sdnNjb2RlLXJlbW90ZQ==.jl:9
